In [ ]:
# Connect to the DuckDB shipped in your Kaggle Dataset.
# If it's missing, load from CSVs into an in-memory DuckDB.

import os, duckdb, pandas as pd, numpy as np
import matplotlib.pyplot as plt

DATA_DIR = "/kaggle/input/music-listening-data-500k-users"  # adjust if your folder is named differently
DB_PATH  = os.path.join(DATA_DIR, "music.duckdb")

con = duckdb.connect(DB_PATH, read_only=True)
print("Connected to:", DB_PATH)

sql = con.sql

In [ ]:
# ===== Cell: Build a manageable co-listening graph (PMI weights) =====
# Compatible with your DuckDB: deterministic sampling via HASH, no PRAGMAs, no ?:: casts.

import numpy as np
import pandas as pd

# --------- knobs ----------
SAMPLE_USERS      = 50_000   # number of users to sample
TOP_ARTISTS       = 100    # keep only the most popular artists in the sample
MIN_FANS          = 100      # minimum sampled fans to keep an artist
MIN_COUSERS       = 50       # keep edges seen by at least this many sampled users
ONLY_POSITIVE_PMI = True     # drop edges with PMI <= 0
RANDOM_SEED       = 30       # used for deterministic sampling
# --------------------------

# Clean any leftovers
for obj in ["sampled_users","ua","fans","top_artists","pairs","artist_sim","artist_neighbors"]:
    try: sql(f"DROP VIEW  IF EXISTS {obj};")
    except: pass
    try: sql(f"DROP TABLE IF EXISTS {obj};")
    except: pass

# 1) Deterministic user sample via hash(user_id || seed)
sql("""
CREATE TEMP TABLE sampled_users AS
SELECT user_id
FROM users
ORDER BY hash(CAST(user_id AS VARCHAR) || ?)
LIMIT ?;
""", params=[str(RANDOM_SEED), SAMPLE_USERS])

# 2) User–Artist pairs within the sample
sql("""
CREATE TEMP TABLE ua AS
SELECT a.user_id, a.artist_name
FROM user_top_artists a
JOIN sampled_users s USING (user_id);
""")

# 3) Fans per artist (in sample)
sql("CREATE TEMP TABLE fans AS SELECT artist_name, COUNT(DISTINCT user_id) AS fans FROM ua GROUP BY 1;")

# 4) Keep top artists by sampled fans (cap)
sql("""
CREATE TEMP TABLE top_artists AS
SELECT artist_name, fans
FROM fans
WHERE fans >= ?
ORDER BY fans DESC
LIMIT ?;
""", params=[MIN_FANS, TOP_ARTISTS])

# 5) Co-occurrence among kept artists
sql("""
CREATE TEMP TABLE pairs AS
SELECT 
  u1.artist_name AS a,
  u2.artist_name AS b,
  COUNT(DISTINCT u1.user_id) AS co_users
FROM ua u1
JOIN ua u2 
  ON u1.user_id = u2.user_id 
 AND u1.artist_name < u2.artist_name
JOIN top_artists t1 ON t1.artist_name = u1.artist_name
JOIN top_artists t2 ON t2.artist_name = u2.artist_name
GROUP BY 1,2
HAVING co_users >= ?;
""", params=[MIN_COUSERS])

# 6) PMI weights
N = sql("SELECT COUNT(DISTINCT user_id) FROM ua;").fetchone()[0]

# Inject N as a bound parameter directly in the formula (no temp table, no :: casts)
sql("""
CREATE TEMP TABLE artist_sim AS
SELECT 
  p.a, p.b, p.co_users,
  f1.fans AS fans_a,
  f2.fans AS fans_b,
  ln( (p.co_users * CAST(? AS DOUBLE)) / (f1.fans * f2.fans) ) AS pmi
FROM pairs p
JOIN fans f1 ON f1.artist_name = p.a
JOIN fans f2 ON f2.artist_name = p.b;
""", params=[N])

if ONLY_POSITIVE_PMI:
    sql("DELETE FROM artist_sim WHERE pmi <= 0;")

# Convenience view (both directions)
sql("""
CREATE TEMP VIEW artist_neighbors AS
SELECT a AS artist, b AS neighbor, pmi, co_users FROM artist_sim
UNION ALL
SELECT b AS artist, a AS neighbor, pmi, co_users FROM artist_sim;
""")

# Materialize nodes/edges for plotting
nodes_df = sql("SELECT artist_name AS artist, fans FROM top_artists;").df()
edges_df = sql("SELECT a AS src, b AS dst, pmi, co_users FROM artist_sim;").df()

print(f"Nodes: {len(nodes_df):,} | Edges: {len(edges_df):,} | Sampled users: {N:,}")
nodes_df.head(), edges_df.head()

In [ ]:
# ===== Cell 3 — Build the network, detect communities, layout (NX 3.x safe) =====
import networkx as nx
import numpy as np
import pandas as pd

# Build graph
G = nx.Graph()
G.add_nodes_from(
    (r.artist, {"fans": int(r.fans)}) for _, r in nodes_df.iterrows()
)
G.add_edges_from(
    (r.src, r.dst, {"weight": float(r.pmi), "co_users": int(r.co_users)})
    for _, r in edges_df.iterrows()
)

# Summary (nx.info was removed in NetworkX 3.x)
print(f"Graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")

# Community detection
try:
    import community as community_louvain  # python-louvain (if available on the image)
    partition = community_louvain.best_partition(
        G, weight="weight", resolution=1.0, random_state=42
    )
except Exception:
    # Fallback: greedy modularity (no resolution parameter)
    comms = list(nx.algorithms.community.greedy_modularity_communities(G, weight="weight"))
    partition = {}
    for i, c in enumerate(comms):
        for n in c:
            partition[n] = i

nx.set_node_attributes(G, partition, "group")

# Layout (spring as a ForceAtlas-like fallback)
n = G.number_of_nodes()
k = 1 / np.sqrt(n) if n > 0 else 0.1
pos = nx.spring_layout(G, k=k, iterations=300, weight="weight", seed=42)

In [ ]:
# ===== Dramatic plot: boost node + edge contrast (with wider layout) =====
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

# --- layout knobs (makes the graph more spread out) ---
SPREAD_FACTOR = 3.5   # larger => more space between nodes (try 2–6)
ITERATIONS    = 1000  # more iterations => smoother layout
# recompute a wider spring layout (ignores any previous `pos`)
n = G.number_of_nodes()
k = SPREAD_FACTOR / np.sqrt(n) if n > 0 else 0.1
pos = nx.spring_layout(G, k=k, iterations=ITERATIONS, weight="weight", seed=42)

# --- visual knobs ---
TOP_LABELS_PER_GROUP = 12
STRONG_EDGE_Q        = 0.70  # edges above this quantile are "strong"
NODE_EXP             = 1.8
NODE_BASE            = 80
NODE_SCALE           = 2200
STRONG_ALPHA         = 0.45
WEAK_ALPHA           = 0.03
# --------------------

plt.figure(figsize=(12, 10), facecolor="black")
ax = plt.gca()
ax.set_facecolor("black")
plt.axis("off")

# Colors by community (fallback to gray if group missing)
groups = pd.Series(nx.get_node_attributes(G, "group"))
uniq = sorted(groups.unique())
palette = plt.cm.tab20(np.linspace(0, 1, max(20, len(uniq))))
color_map = {g: palette[i % len(palette)] for i, g in enumerate(uniq)}
node_colors = [color_map.get(groups.get(n_, -1), (0.7, 0.7, 0.7, 1)) for n_ in G.nodes()]

# Node sizes (percentile-normalized + nonlinear)
fans = pd.Series(nx.get_node_attributes(G, "fans")).reindex(G.nodes()).fillna(1).astype(float).values
p10, p90 = np.percentile(fans, [10, 90]) if len(fans) else (0, 1)
scaled = np.clip((fans - p10) / (p90 - p10 + 1e-9), 0, 1)
node_sizes = NODE_BASE + NODE_SCALE * (scaled ** NODE_EXP)

# Edge layers
edges = list(G.edges(data=True))
w = np.array([attr.get("weight", 0.0) for _, _, attr in edges], dtype=float)
if len(w):
    wq = np.quantile(w, STRONG_EDGE_Q)
    strong_idx = w >= wq
    weak_idx   = ~strong_idx

    # Weak hairlines
    if weak_idx.any():
        nx.draw_networkx_edges(
            G, pos,
            edgelist=[(u, v) for (i, (u, v, _)) in enumerate(edges) if weak_idx[i]],
            width=0.2, edge_color="white", alpha=WEAK_ALPHA
        )

    # Strong strokes with nonlinear width
    if strong_idx.any():
        w_strong = w[strong_idx]
        w_min, w_max = w_strong.min(), w_strong.max()
        w_norm = (w_strong - w_min) / (w_max - w_min + 1e-9)
        widths = 1.8 + 6.5 * (w_norm ** 1.7)
        nx.draw_networkx_edges(
            G, pos,
            edgelist=[(u, v) for (i, (u, v, _)) in enumerate(edges) if strong_idx[i]],
            width=list(widths), edge_color="white", alpha=STRONG_ALPHA
        )

# Draw nodes on top
nx.draw_networkx_nodes(
    G, pos, node_color=node_colors, node_size=node_sizes, linewidths=0, alpha=0.95
)

# Labels: show all artist names
for node in G.nodes():
    x, y = pos[node]
    plt.text(
        x, y, node,
        fontsize=8, color="white",
        ha="center", va="center", alpha=0.95
    )

plt.title("Artist Co-Listening Network (PMI-weighted, emphasized)", color="white")
plt.tight_layout()
out_png = "/kaggle/working/artist_graph_emphasized.png"
plt.savefig(out_png, dpi=1000, facecolor="black")
print("Saved:", out_png)
plt.show()

In [ ]:
# Write a GEXF you can open in Gephi for high-quality ForceAtlas2 / labeling
out_gexf = "/kaggle/working/artist_graph.gexf"
nx.write_gexf(G, out_gexf)
print("Wrote:", out_gexf)